# Impact estimator

The impact estimator predicts, from the report alone, whether a new emergency is likely to be high impact. This notebook compares a logistic regression, a gradient boosting model and a simple type-and-location baseline on the most recent months, and shows the factors behind individual predictions.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## Temporal split

Models are tested on the last three months of data, never on shuffled rows. `split_dates` returns the start of the validation and test periods, and `temporal_masks` selects the rows of each period.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_curve

from ml.data import load_frame
from ml.evaluation import split_dates, temporal_masks
from ml.features import report_time_features
from ml.impact import (
    GRADIENT_BOOSTING,
    LOGISTIC_REGRESSION,
    TypeLocationBaseline,
    fit_pipeline,
    train_impact,
)

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
split = split_dates(frame["occurred_on"])
masks = temporal_masks(frame, split)
print(f"Validation starts: {split.validation_start}")
print(f"Test starts: {split.test_start}")
print({name: int(mask.sum()) for name, mask in masks.items()})

## Train and compare

`train_impact` trains the three candidates, picks the decision threshold on the validation period and scores each candidate on the test period. The table reports the F1 score of each candidate on the test rows.

In [ ]:
result = train_impact(frame)
metrics = result.metrics
comparison = pd.DataFrame(
    {
        "F1": {
            "Logistic regression": metrics["logistic_regression_f1"],
            "Gradient boosting": metrics["gradient_boosting_f1"],
            "Baseline": metrics["baseline_f1"],
        }
    }
).round(3)
print(f"Chosen model: {metrics['model']}")
print(f"Test rows: {metrics['test_rows']}, high-impact rows: {metrics['test_positives']}")
comparison

## Precision-recall curves

Each curve is computed on the test period by a model fitted on all earlier rows. A curve that stays higher means the model finds more high-impact emergencies for the same number of false alarms.

In [ ]:
train, test = frame[masks["train"]], frame[masks["test"]]
labels = test["is_high_impact"].to_numpy(dtype=bool)
curves = {
    "Logistic regression": fit_pipeline(LOGISTIC_REGRESSION, train)
    .predict_proba(report_time_features(test))[:, 1],
    "Gradient boosting": fit_pipeline(GRADIENT_BOOSTING, train)
    .predict_proba(report_time_features(test))[:, 1],
    "Baseline": TypeLocationBaseline().fit(train).score(test),
}
fig, ax = plt.subplots(figsize=(7, 5))
for name, scores in curves.items():
    precision, recall, _ = precision_recall_curve(labels, scores)
    ax.plot(recall, precision, label=name)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.show()

## Explain predictions

`explain` lists the three factors that weigh most in a prediction. The model is applied here to three high-impact emergencies from the test period.

In [ ]:
model = result.model
examples = test[test["is_high_impact"]].head(3)
probabilities = model.probability(examples)
for (_, row), probability, factors in zip(
    examples.iterrows(), probabilities, model.explain(examples)
):
    print(f"{row['event_type']} on {row['occurred_on']}: probability {probability:.2f}")
    for factor in factors:
        print(f"  {factor.feature}: {factor.value} ({factor.weight:+.2f})")